# Cell 01 - Load dữ liệu đã xử lý từ notebook 01

## Mục tiêu cell này
Đọc lại các file dữ liệu đã tạo ở notebook `01_download_and_inspect_data.ipynb`.

## Giải thích code
Code sẽ load:
- `legal_corpus.csv`: kho văn bản pháp luật tiếng Việt
- `train_queries.csv`: câu hỏi train và relevant cid
- `test_queries.csv`: câu hỏi test và relevant cid
- `company_handbook_documents.csv`: tài liệu handbook nội bộ công ty

Đồng thời chuyển cột `relevant_cids` từ dạng chuỗi JSON về list Python để các bước kiểm tra leakage sau dễ xử lý.

## Output mong đợi
Bạn cần thấy:
- Legal corpus khoảng 68,663 dòng
- Train queries khoảng 89,261 dòng
- Test queries khoảng 29,746 dòng
- Company handbook 16 dòng

In [1]:
from pathlib import Path
import pandas as pd
import json

cwd = Path.cwd().resolve()
PROJECT = cwd.parent if cwd.name == "notebooks" else cwd

PROCESSED_DIR = PROJECT / "data" / "processed"
SPLIT_DIR = PROJECT / "data" / "splits"
REPORT_DIR = PROJECT / "artifacts" / "leakage_reports"

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

legal_corpus_df = pd.read_csv(PROCESSED_DIR / "legal_corpus.csv")
train_queries_df = pd.read_csv(PROCESSED_DIR / "train_queries.csv")
test_queries_df = pd.read_csv(PROCESSED_DIR / "test_queries.csv")
company_docs_df = pd.read_csv(PROCESSED_DIR / "company_handbook_documents.csv")

train_queries_df["relevant_cids"] = train_queries_df["relevant_cids"].apply(json.loads)
test_queries_df["relevant_cids"] = test_queries_df["relevant_cids"].apply(json.loads)

print("Project:", PROJECT)
print("Legal corpus:", legal_corpus_df.shape)
print("Train queries:", train_queries_df.shape)
print("Test queries:", test_queries_df.shape)
print("Company handbook:", company_docs_df.shape)

display(train_queries_df.head())

Project: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag
Legal corpus: (68663, 3)
Train queries: (89261, 4)
Test queries: (29746, 4)
Company handbook: (16, 6)


,qid,question,relevant_cids,num_relevant
0,72600,Liên đoàn Luật sư Việt Nam là tổ chức xã hội –...,[142820],1
1,147562,Tên hợp tác xã bị rơi vào trường hợp cấm thì c...,"[27817, 72117]",2
2,142107,Tài xế lái xe ô tô khách 50 chỗ ngồi bao lâu t...,"[33215, 56201]",2
3,77353,Các bước chuẩn bị thủ thuật bó bột Cravate sẽ ...,[148158],1
4,113090,Viên chức Hộ sinh hạng 4 có những nhiệm vụ gì ...,[188132],1


# Cell 02 - Kiểm tra trùng câu hỏi giữa train và test

## Mục tiêu cell này
Kiểm tra xem có câu hỏi nào bị trùng chính xác giữa `train_queries.csv` và `test_queries.csv` không.

## Giải thích code
Code sẽ chuẩn hóa câu hỏi bằng cách:
- chuyển về chữ thường
- xóa khoảng trắng thừa
- so sánh câu hỏi đã chuẩn hóa giữa train và test

Sau đó lưu báo cáo vào:
`artifacts/leakage_reports/exact_question_overlap_train_test.csv`

## Output mong đợi
Nếu tốt, số câu hỏi overlap giữa train và test nên bằng 0.  
Nếu lớn hơn 0, ta sẽ ghi nhận trong leakage report và xử lý ở bước sau.

In [2]:
import re
import pandas as pd

def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

train_queries_df["question_norm"] = train_queries_df["question"].apply(normalize_text)
test_queries_df["question_norm"] = test_queries_df["question"].apply(normalize_text)

train_counts = train_queries_df["question_norm"].value_counts().reset_index()
test_counts = test_queries_df["question_norm"].value_counts().reset_index()

train_counts.columns = ["question_norm", "train_count"]
test_counts.columns = ["question_norm", "test_count"]

overlap_df = train_counts.merge(test_counts, on="question_norm", how="inner")

report_path = REPORT_DIR / "exact_question_overlap_train_test.csv"
overlap_df.to_csv(report_path, index=False, encoding="utf-8-sig")

print("Unique train questions:", train_queries_df["question_norm"].nunique())
print("Unique test questions:", test_queries_df["question_norm"].nunique())
print("Exact overlap questions:", len(overlap_df))
print("Train rows affected:", overlap_df["train_count"].sum() if len(overlap_df) else 0)
print("Test rows affected:", overlap_df["test_count"].sum() if len(overlap_df) else 0)
print("Đã lưu report:", report_path)

display(overlap_df.head(10))

Unique train questions: 87884
Unique test questions: 29573
Exact overlap questions: 723
Train rows affected: 1017
Test rows affected: 797
Đã lưu report: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\leakage_reports\exact_question_overlap_train_test.csv


,question_norm,train_count,test_count
0,quỹ bình ổn giá xăng dầu được thực hiện như th...,14,11
1,thương nhân kinh doanh dịch vụ xăng dầu có quy...,10,4
2,thông tư 04/2023/tt-btp được áp dụng từ ngày mấy?,10,2
3,ký kết thỏa thuận quốc tế là gì?,9,2
4,viên chức là ai?,9,1
5,hóa đơn điện tử là gì?,8,1
6,nghị định 31/2023/nđ-cp có hiệu lực từ ngày nào?,7,3
7,thông tư 24/2023/tt-bca về đăng ký xe được chí...,7,4
8,ký kết thỏa thuận quốc tế dựa trên những nguyê...,7,2
9,thuế giá trị gia tăng là gì?,6,1


# Cell 03 - Loại câu hỏi train bị trùng chính xác với test

## Mục tiêu cell này
Tạo tập train sạch hơn bằng cách loại bỏ các dòng train có câu hỏi trùng chính xác với test.

## Giải thích code
Code sẽ:
- lấy danh sách `question_norm` xuất hiện ở cả train và test
- xóa các dòng train có `question_norm` nằm trong danh sách overlap
- giữ nguyên test set để dùng đánh giá cuối
- lưu train đã làm sạch vào `data/splits/train_no_exact_question_overlap.csv`

Cách này giúp giảm leakage vì test không còn xuất hiện lại trong dữ liệu dùng để tune pipeline.

## Output mong đợi
Bạn cần thấy:
- số dòng train ban đầu
- số dòng train bị loại
- số dòng train còn lại
- exact overlap sau khi lọc bằng 0

In [3]:
overlap_questions = set(overlap_df["question_norm"])

train_clean_df = train_queries_df[~train_queries_df["question_norm"].isin(overlap_questions)].copy()
removed_df = train_queries_df[train_queries_df["question_norm"].isin(overlap_questions)].copy()

clean_path = SPLIT_DIR / "train_no_exact_question_overlap.csv"
removed_path = REPORT_DIR / "removed_train_exact_question_overlap.csv"

save_cols = ["qid", "question", "relevant_cids", "num_relevant"]

train_clean_save = train_clean_df[save_cols].copy()
removed_save = removed_df[save_cols + ["question_norm"]].copy()

train_clean_save["relevant_cids"] = train_clean_save["relevant_cids"].apply(lambda xs: json.dumps(xs, ensure_ascii=False))
removed_save["relevant_cids"] = removed_save["relevant_cids"].apply(lambda xs: json.dumps(xs, ensure_ascii=False))

train_clean_save.to_csv(clean_path, index=False, encoding="utf-8-sig")
removed_save.to_csv(removed_path, index=False, encoding="utf-8-sig")

remaining_overlap = set(train_clean_df["question_norm"]) & set(test_queries_df["question_norm"])

print("Train ban đầu:", len(train_queries_df))
print("Số dòng train bị loại:", len(removed_df))
print("Train còn lại:", len(train_clean_df))
print("Exact overlap còn lại:", len(remaining_overlap))
print("Đã lưu train sạch:", clean_path)
print("Đã lưu report dòng bị loại:", removed_path)

display(train_clean_df[save_cols].head())

Train ban đầu: 89261
Số dòng train bị loại: 1017
Train còn lại: 88244
Exact overlap còn lại: 0
Đã lưu train sạch: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\data\splits\train_no_exact_question_overlap.csv
Đã lưu report dòng bị loại: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\leakage_reports\removed_train_exact_question_overlap.csv


,qid,question,relevant_cids,num_relevant
0,72600,Liên đoàn Luật sư Việt Nam là tổ chức xã hội –...,[142820],1
1,147562,Tên hợp tác xã bị rơi vào trường hợp cấm thì c...,"[27817, 72117]",2
2,142107,Tài xế lái xe ô tô khách 50 chỗ ngồi bao lâu t...,"[33215, 56201]",2
3,77353,Các bước chuẩn bị thủ thuật bó bột Cravate sẽ ...,[148158],1
4,113090,Viên chức Hộ sinh hạng 4 có những nhiệm vụ gì ...,[188132],1


# Cell 04 - Kiểm tra duplicate question bên trong từng split

## Mục tiêu cell này
Kiểm tra các câu hỏi bị trùng lặp bên trong:
- train đã làm sạch
- test set

## Giải thích code
Code sẽ đếm số lần xuất hiện của mỗi `question_norm` trong từng split.  
Nếu một câu hỏi xuất hiện nhiều lần, nó sẽ được lưu vào report để sau này chia validation theo nhóm câu hỏi, tránh cùng một câu hỏi rơi vào cả train và validation.

## Output mong đợi
Bạn cần thấy số lượng câu hỏi duplicate trong train/test.  
Có duplicate không nhất thiết là lỗi nghiêm trọng, nhưng khi chia validation phải xử lý cẩn thận.

In [4]:
train_dup_questions = (
    train_clean_df["question_norm"]
    .value_counts()
    .reset_index()
)

test_dup_questions = (
    test_queries_df["question_norm"]
    .value_counts()
    .reset_index()
)

train_dup_questions.columns = ["question_norm", "count"]
test_dup_questions.columns = ["question_norm", "count"]

train_dup_questions = train_dup_questions[train_dup_questions["count"] > 1].reset_index(drop=True)
test_dup_questions = test_dup_questions[test_dup_questions["count"] > 1].reset_index(drop=True)

train_dup_path = REPORT_DIR / "duplicate_questions_within_train_clean.csv"
test_dup_path = REPORT_DIR / "duplicate_questions_within_test.csv"

train_dup_questions.to_csv(train_dup_path, index=False, encoding="utf-8-sig")
test_dup_questions.to_csv(test_dup_path, index=False, encoding="utf-8-sig")

print("Duplicate questions trong train clean:", len(train_dup_questions))
print("Duplicate questions trong test:", len(test_dup_questions))
print("Đã lưu train duplicate report:", train_dup_path)
print("Đã lưu test duplicate report:", test_dup_path)

print("\nTop duplicate train questions:")
display(train_dup_questions.head(10))

print("\nTop duplicate test questions:")
display(test_dup_questions.head(10))

Duplicate questions trong train clean: 945
Duplicate questions trong test: 150
Đã lưu train duplicate report: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\leakage_reports\duplicate_questions_within_train_clean.csv
Đã lưu test duplicate report: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\leakage_reports\duplicate_questions_within_test.csv

Top duplicate train questions:


,question_norm,count
0,hợp đồng lao động là gì?,9
1,hội đồng quản trị là gì?,8
2,tai nạn lao động là gì?,6
3,điểm liệt thi tốt nghiệp thpt năm 2023 là bao ...,5
4,hộ kinh doanh là gì?,5
5,mức hưởng chế độ thai sản được quy định như th...,5
6,công trình xây dựng là gì?,5
7,thí sinh nhận giấy báo nhập học đại học tại đâu?,5
8,nhiệm vụ của hội môi trường đô thị việt nam là...,4
9,công ty cổ phần là gì?,4



Top duplicate test questions:


,question_norm,count
0,quỹ bình ổn giá xăng dầu được thực hiện như th...,11
1,thương nhân kinh doanh dịch vụ xăng dầu có quy...,4
2,thông tư 24/2023/tt-bca về đăng ký xe được chí...,4
3,viên chức là gì?,4
4,chứng chỉ hành nghề xây dựng là gì?,4
5,"đối tượng học sinh, sinh viên nào sẽ được hỗ t...",3
6,khi nào nghị định 47/2023/nđ-cp sửa đổi nghị đ...,3
7,nghị định 31/2023/nđ-cp có hiệu lực từ ngày nào?,3
8,tuổi của học sinh các cấp là bao nhiêu?,3
9,người lao động làm việc vào ngày lễ quốc khánh...,3


Kết quả này cho thấy trong train vẫn có nhiều câu hỏi lặp lại. Không sao, nhưng khi chia validation thì không được chia theo từng dòng ngẫu nhiên, vì cùng một câu hỏi có thể rơi vào cả train và validation. Ta sẽ chia theo nhóm question_norm

# Cell 05 - Chia train/validation theo nhóm câu hỏi để tránh leakage

## Mục tiêu cell này
Tạo `train_final.csv` và `val_queries.csv` từ train đã làm sạch.

## Giải thích code
Code sẽ chia dữ liệu theo `question_norm`, không chia theo từng dòng.  
Như vậy, nếu một câu hỏi bị lặp nhiều lần, toàn bộ các dòng đó chỉ nằm trong train hoặc validation, không bị rơi vào cả hai.

Test set được giữ nguyên và chưa dùng để tune pipeline.

## Output mong đợi
Bạn cần thấy:
- số dòng train final
- số dòng validation
- overlap question giữa train final và validation bằng 0
- overlap question giữa validation và test bằng 0

In [5]:
from sklearn.model_selection import train_test_split
import json

unique_questions = train_clean_df["question_norm"].drop_duplicates()

train_norms, val_norms = train_test_split(
    unique_questions,
    test_size=0.1,
    random_state=42,
    shuffle=True
)

train_final_df = train_clean_df[train_clean_df["question_norm"].isin(set(train_norms))].copy()
val_queries_df = train_clean_df[train_clean_df["question_norm"].isin(set(val_norms))].copy()

train_val_overlap = set(train_final_df["question_norm"]) & set(val_queries_df["question_norm"])
val_test_overlap = set(val_queries_df["question_norm"]) & set(test_queries_df["question_norm"])

save_cols = ["qid", "question", "relevant_cids", "num_relevant"]

train_final_save = train_final_df[save_cols].copy()
val_save = val_queries_df[save_cols].copy()

train_final_save["relevant_cids"] = train_final_save["relevant_cids"].apply(lambda xs: json.dumps(xs, ensure_ascii=False))
val_save["relevant_cids"] = val_save["relevant_cids"].apply(lambda xs: json.dumps(xs, ensure_ascii=False))

train_final_path = SPLIT_DIR / "train_final.csv"
val_path = SPLIT_DIR / "val_queries.csv"

train_final_save.to_csv(train_final_path, index=False, encoding="utf-8-sig")
val_save.to_csv(val_path, index=False, encoding="utf-8-sig")

print("Train final rows:", len(train_final_df))
print("Validation rows:", len(val_queries_df))
print("Train/Val question overlap:", len(train_val_overlap))
print("Val/Test question overlap:", len(val_test_overlap))
print("Đã lưu:", train_final_path)
print("Đã lưu:", val_path)

display(val_save.head())

Train final rows: 79424
Validation rows: 8820
Train/Val question overlap: 0
Val/Test question overlap: 0
Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\data\splits\train_final.csv
Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\data\splits\val_queries.csv


,qid,question,relevant_cids,num_relevant
40,162228,Chế độ báo cáo của Giám đốc Ban Quản lý dự án ...,[243090],1
78,3414,Việc tích nước vận hành và tích nước thời kỳ t...,[65231],1
104,108965,Phanh cần trục khi dừng khẩn cấp phải là loại ...,[183501],1
106,103161,Ai có trách nhiệm báo cáo kết quả đánh giá ngh...,[177010],1
109,70371,Làm sao để được công nhận văn bằng nước ngoài ...,"[140295, 140296]",2


# Cell 06 - Kiểm tra overlap relevant cid giữa train, validation và test

## Mục tiêu cell này
Kiểm tra xem các tài liệu pháp luật đúng (`relevant_cids`) có bị xuất hiện lặp giữa train, validation và test không.

## Giải thích code
Code sẽ:
- tách danh sách `relevant_cids` thành từng cid riêng
- tạo tập cid cho train, validation và test
- tính số cid bị overlap giữa các split
- lưu báo cáo vào `artifacts/leakage_reports/cid_overlap_between_splits.csv`

Lưu ý: Với bài toán RAG, corpus tài liệu có thể dùng chung cho mọi split. Tuy nhiên, nếu sau này fine-tune reranker/verifier bằng nhãn train, việc document overlap cần được ghi nhận rõ trong báo cáo leakage.

## Output mong đợi
Bạn sẽ thấy số lượng unique relevant cid trong từng split và số cid overlap giữa các cặp split.

In [6]:
def get_cid_set(df):
    return set(cid for cids in df["relevant_cids"] for cid in cids)

train_cids = get_cid_set(train_final_df)
val_cids = get_cid_set(val_queries_df)
test_cids = get_cid_set(test_queries_df)

overlap_rows = [
    {
        "split_pair": "train_final_vs_val",
        "overlap_cids": len(train_cids & val_cids),
        "left_unique_cids": len(train_cids),
        "right_unique_cids": len(val_cids)
    },
    {
        "split_pair": "train_final_vs_test",
        "overlap_cids": len(train_cids & test_cids),
        "left_unique_cids": len(train_cids),
        "right_unique_cids": len(test_cids)
    },
    {
        "split_pair": "val_vs_test",
        "overlap_cids": len(val_cids & test_cids),
        "left_unique_cids": len(val_cids),
        "right_unique_cids": len(test_cids)
    }
]

cid_overlap_df = pd.DataFrame(overlap_rows)

cid_report_path = REPORT_DIR / "cid_overlap_between_splits.csv"
cid_overlap_df.to_csv(cid_report_path, index=False, encoding="utf-8-sig")

print("Train unique relevant cids:", len(train_cids))
print("Validation unique relevant cids:", len(val_cids))
print("Test unique relevant cids:", len(test_cids))
print("Đã lưu report:", cid_report_path)

display(cid_overlap_df)

Train unique relevant cids: 51628
Validation unique relevant cids: 8270
Test unique relevant cids: 23907
Đã lưu report: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\leakage_reports\cid_overlap_between_splits.csv


,split_pair,overlap_cids,left_unique_cids,right_unique_cids
0,train_final_vs_val,4170,51628,8270
1,train_final_vs_test,10494,51628,23907
2,val_vs_test,2698,8270,23907


# Cell 07 - Tạo báo cáo tổng hợp leakage

## Mục tiêu cell này
Tổng hợp các kết quả kiểm tra leakage đã làm thành một bảng báo cáo.

## Giải thích code
Code sẽ gom các thông tin:
- số dòng train ban đầu
- số dòng train sau khi loại exact question overlap với test
- số dòng train/validation/test cuối cùng
- số exact question overlap sau xử lý
- số duplicate question trong từng split
- số overlap relevant cid giữa các split

Báo cáo được lưu vào:
`artifacts/leakage_reports/leakage_summary.csv`

## Output mong đợi
Bạn cần thấy bảng tổng hợp các chỉ số leakage để dùng trong báo cáo đồ án.

In [7]:
summary = {
    "original_train_rows": len(train_queries_df),
    "train_rows_removed_exact_question_overlap_with_test": len(removed_df),
    "train_final_rows": len(train_final_df),
    "validation_rows": len(val_queries_df),
    "test_rows": len(test_queries_df),
    "train_test_exact_question_overlap_after_cleaning": len(set(train_clean_df["question_norm"]) & set(test_queries_df["question_norm"])),
    "train_final_val_question_overlap": len(set(train_final_df["question_norm"]) & set(val_queries_df["question_norm"])),
    "val_test_question_overlap": len(set(val_queries_df["question_norm"]) & set(test_queries_df["question_norm"])),
    "duplicate_questions_train_clean": len(train_dup_questions),
    "duplicate_questions_test": len(test_dup_questions),
    "cid_overlap_train_val": len(train_cids & val_cids),
    "cid_overlap_train_test": len(train_cids & test_cids),
    "cid_overlap_val_test": len(val_cids & test_cids)
}

leakage_summary_df = pd.DataFrame([summary])

summary_path = REPORT_DIR / "leakage_summary.csv"
leakage_summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("Đã lưu:", summary_path)
display(leakage_summary_df.T.rename(columns={0: "value"}))

Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\leakage_reports\leakage_summary.csv


,value
original_train_rows,89261
train_rows_removed_exact_question_overlap_with_test,1017
train_final_rows,79424
validation_rows,8820
test_rows,29746
train_test_exact_question_overlap_after_cleaning,0
train_final_val_question_overlap,0
val_test_question_overlap,0
duplicate_questions_train_clean,945
duplicate_questions_test,150


# Cell 08 - Kiểm tra coverage của relevant cid trong legal corpus

## Mục tiêu cell này
Kiểm tra toàn bộ `relevant_cids` trong train, validation và test có tồn tại trong `legal_corpus.csv` hay không.

## Giải thích code
Code sẽ:
- lấy toàn bộ `cid` trong legal corpus
- lấy toàn bộ `relevant_cids` của train, validation và test
- kiểm tra số lượng cid bị thiếu khỏi corpus
- lưu báo cáo vào `artifacts/leakage_reports/cid_coverage_report.csv`

Nếu có cid bị thiếu, retriever sẽ không thể tìm đúng evidence cho các câu hỏi đó.

## Output mong đợi
Số `missing_cids` của train, validation và test nên bằng 0.

In [8]:
legal_corpus_cids = set(legal_corpus_df["cid"].astype(int))

def cid_coverage(split_name, df):
    cids = get_cid_set(df)
    missing = sorted(cids - legal_corpus_cids)
    return {
        "split": split_name,
        "unique_relevant_cids": len(cids),
        "missing_cids": len(missing),
        "coverage_rate": round((len(cids) - len(missing)) / len(cids), 6) if len(cids) else 0,
        "missing_cid_examples": missing[:10]
    }

coverage_df = pd.DataFrame([
    cid_coverage("train_final", train_final_df),
    cid_coverage("validation", val_queries_df),
    cid_coverage("test", test_queries_df)
])

coverage_path = REPORT_DIR / "cid_coverage_report.csv"
coverage_df.to_csv(coverage_path, index=False, encoding="utf-8-sig")

print("Đã lưu:", coverage_path)
display(coverage_df)

Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\leakage_reports\cid_coverage_report.csv


,split,unique_relevant_cids,missing_cids,coverage_rate,missing_cid_examples
0,train_final,51628,0,1.0,[]
1,validation,8270,0,1.0,[]
2,test,23907,0,1.0,[]


# Cell 09 - Kiểm tra overlap qid giữa train, validation và test

## Mục tiêu cell này
Kiểm tra `qid` có bị trùng giữa các split train, validation và test không.

## Giải thích code
Code sẽ:
- lấy tập `qid` của train final, validation và test
- kiểm tra overlap giữa từng cặp split
- kiểm tra duplicate `qid` bên trong từng split
- lưu báo cáo vào `artifacts/leakage_reports/qid_overlap_report.csv`

Nếu `qid` overlap giữa các split lớn hơn 0, cần xử lý vì cùng một câu hỏi/mẫu dữ liệu có thể bị dùng ở nhiều tập.

## Output mong đợi
Các giá trị overlap giữa train/validation/test nên bằng 0.

In [9]:
def qid_report(split_name, df):
    return {
        "split": split_name,
        "rows": len(df),
        "unique_qid": df["qid"].nunique(),
        "duplicate_qid_rows": len(df) - df["qid"].nunique()
    }

train_qids = set(train_final_df["qid"])
val_qids = set(val_queries_df["qid"])
test_qids = set(test_queries_df["qid"])

qid_overlap_rows = [
    qid_report("train_final", train_final_df),
    qid_report("validation", val_queries_df),
    qid_report("test", test_queries_df),
    {
        "split": "overlap_train_val",
        "rows": len(train_qids & val_qids),
        "unique_qid": len(train_qids & val_qids),
        "duplicate_qid_rows": 0
    },
    {
        "split": "overlap_train_test",
        "rows": len(train_qids & test_qids),
        "unique_qid": len(train_qids & test_qids),
        "duplicate_qid_rows": 0
    },
    {
        "split": "overlap_val_test",
        "rows": len(val_qids & test_qids),
        "unique_qid": len(val_qids & test_qids),
        "duplicate_qid_rows": 0
    }
]

qid_overlap_df = pd.DataFrame(qid_overlap_rows)

qid_report_path = REPORT_DIR / "qid_overlap_report.csv"
qid_overlap_df.to_csv(qid_report_path, index=False, encoding="utf-8-sig")

print("Đã lưu:", qid_report_path)
display(qid_overlap_df)

Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\leakage_reports\qid_overlap_report.csv


,split,rows,unique_qid,duplicate_qid_rows
0,train_final,79424,79424,0
1,validation,8820,8820,0
2,test,29746,29746,0
3,overlap_train_val,0,0,0
4,overlap_train_test,0,0,0
5,overlap_val_test,0,0,0


# Cell 10 - Kiểm tra near-duplicate question giữa các split

## Mục tiêu cell này
Kiểm tra các câu hỏi gần giống nhau giữa:
- train_final và test
- validation và test
- train_final và validation

## Giải thích code
Code dùng TF-IDF ký tự n-gram để biểu diễn câu hỏi, sau đó tìm câu hỏi gần nhất giữa các split bằng cosine similarity.  
Nếu similarity rất cao, ví dụ từ 0.95 trở lên, ta xem đó là near-duplicate cần ghi nhận vào leakage report.

## Output mong đợi
Bạn cần thấy số lượng near-duplicate theo từng cặp split và file report được lưu trong `artifacts/leakage_reports`.

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
import pandas as pd

NEAR_DUP_THRESHOLD = 0.95

def find_near_duplicates(left_df, right_df, left_name, right_name, threshold=0.95):
    left_texts = left_df["question_norm"].tolist()
    right_texts = right_df["question_norm"].tolist()

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_features=100000)
    vectorizer.fit(left_texts + right_texts)

    left_vec = vectorizer.transform(left_texts)
    right_vec = vectorizer.transform(right_texts)

    nn = NearestNeighbors(n_neighbors=1, metric="cosine", algorithm="brute")
    nn.fit(right_vec)

    distances, indices = nn.kneighbors(left_vec)
    sims = 1 - distances.ravel()
    nearest_idx = indices.ravel()

    rows = []
    for i, sim in enumerate(sims):
        if sim >= threshold:
            rows.append({
                "left_split": left_name,
                "right_split": right_name,
                "similarity": float(sim),
                "left_qid": left_df.iloc[i]["qid"],
                "right_qid": right_df.iloc[nearest_idx[i]]["qid"],
                "left_question": left_df.iloc[i]["question"],
                "right_question": right_df.iloc[nearest_idx[i]]["question"]
            })

    return pd.DataFrame(rows)

near_train_test_df = find_near_duplicates(train_final_df, test_queries_df, "train_final", "test", NEAR_DUP_THRESHOLD)
near_val_test_df = find_near_duplicates(val_queries_df, test_queries_df, "validation", "test", NEAR_DUP_THRESHOLD)
near_train_val_df = find_near_duplicates(train_final_df, val_queries_df, "train_final", "validation", NEAR_DUP_THRESHOLD)

near_all_df = pd.concat([near_train_test_df, near_val_test_df, near_train_val_df], ignore_index=True)

near_report_path = REPORT_DIR / "near_duplicate_questions_between_splits.csv"
near_all_df.to_csv(near_report_path, index=False, encoding="utf-8-sig")

print("Near-duplicate threshold:", NEAR_DUP_THRESHOLD)
print("Near train/test:", len(near_train_test_df))
print("Near val/test:", len(near_val_test_df))
print("Near train/val:", len(near_train_val_df))
print("Đã lưu report:", near_report_path)

display(near_all_df.head(10))

Near-duplicate threshold: 0.95
Near train/test: 612
Near val/test: 64
Near train/val: 181
Đã lưu report: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\leakage_reports\near_duplicate_questions_between_splits.csv


,left_split,right_split,similarity,left_qid,right_qid,left_question,right_question
0,train_final,test,0.976267,114057,129798,"Từ 10/6/2022, chức danh bác sĩ y học dự phòng ...","Từ 10/6/2022, chức danh bác sĩ chính (hạng II)..."
1,train_final,test,0.961666,124085,124084,Quy trình đưa biểu ngữ chúc mừng nhân dịp ngày...,Biểu ngữ chúc mừng nhân dịp ngày Giỗ Tổ Hùng V...
2,train_final,test,0.968882,84463,70890,Phó Chủ tịch Hội đồng đạo đức trong nghiên cứu...,Chủ tịch Hội đồng đạo đức trong nghiên cứu y s...
3,train_final,test,0.974031,125347,81883,Báo cáo kết quả giám sát hoạt động của Đoàn th...,Báo cáo kết quả giám sát hoạt động của Đoàn th...
4,train_final,test,0.976115,118559,129172,Điều kiện để đăng ký kết hôn như thế nào?,Điều kiện để được đăng ký kết hôn như thế nào?
5,train_final,test,0.997678,21351,64490,Chương trình môn học Giáo dục quốc phòng và an...,Chương trình môn học Giáo dục quốc phòng và an...
6,train_final,test,0.950791,75413,11510,Mức hưởng bảo hiểm xã hội một lần được quy địn...,Mức hưởng đối với bảo hiểm xã hội một lần được...
7,train_final,test,0.959445,108760,80105,Thời hạn tạm hoãn xuất cảnh đối với người bị k...,Thời hạn tạm hoãn xuất cảnh đối với người bị k...
8,train_final,test,0.953140,76519,102547,Thuế giá trị gia tăng là gì? Người nộp thuế gi...,Người nộp thuế giá trị gia tăng là ai?
9,train_final,test,0.951445,71017,71018,Cơ quan thường trực giúp việc cho Hội đồng và ...,Cơ quan thường trực giúp việc cho Hội đồng và ...


# Cell 11 - Tạo strict split sau khi loại near-duplicate question

## Mục tiêu cell này
Tạo phiên bản train/validation chặt chẽ hơn bằng cách loại các câu hỏi near-duplicate giữa các split.

## Giải thích code
Code sẽ:
- loại khỏi train các câu gần giống test
- loại khỏi validation các câu gần giống test
- loại khỏi validation các câu gần giống train
- giữ nguyên test set
- lưu kết quả thành `train_strict.csv` và `val_strict.csv`

Ngưỡng near-duplicate đang dùng là cosine similarity >= 0.95 theo TF-IDF character n-gram.

## Output mong đợi
Bạn cần thấy:
- số dòng train trước/sau khi lọc
- số dòng validation trước/sau khi lọc
- số dòng bị loại khỏi từng split
- đường dẫn file strict split đã lưu

In [11]:
train_remove_qids = set(near_train_test_df["left_qid"])
val_remove_qids = set(near_val_test_df["left_qid"]) | set(near_train_val_df["right_qid"])

train_strict_df = train_final_df[~train_final_df["qid"].isin(train_remove_qids)].copy()
val_strict_df = val_queries_df[~val_queries_df["qid"].isin(val_remove_qids)].copy()

train_strict_save = train_strict_df[save_cols].copy()
val_strict_save = val_strict_df[save_cols].copy()

train_strict_save["relevant_cids"] = train_strict_save["relevant_cids"].apply(lambda xs: json.dumps(xs, ensure_ascii=False))
val_strict_save["relevant_cids"] = val_strict_save["relevant_cids"].apply(lambda xs: json.dumps(xs, ensure_ascii=False))

train_strict_path = SPLIT_DIR / "train_strict.csv"
val_strict_path = SPLIT_DIR / "val_strict.csv"

train_strict_save.to_csv(train_strict_path, index=False, encoding="utf-8-sig")
val_strict_save.to_csv(val_strict_path, index=False, encoding="utf-8-sig")

print("Train final trước lọc:", len(train_final_df))
print("Train strict sau lọc:", len(train_strict_df))
print("Train bị loại:", len(train_final_df) - len(train_strict_df))

print("Validation trước lọc:", len(val_queries_df))
print("Validation strict sau lọc:", len(val_strict_df))
print("Validation bị loại:", len(val_queries_df) - len(val_strict_df))

print("Đã lưu:", train_strict_path)
print("Đã lưu:", val_strict_path)

display(val_strict_save.head())

Train final trước lọc: 79424
Train strict sau lọc: 78812
Train bị loại: 612
Validation trước lọc: 8820
Validation strict sau lọc: 8605
Validation bị loại: 215
Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\data\splits\train_strict.csv
Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\data\splits\val_strict.csv


,qid,question,relevant_cids,num_relevant
40,162228,Chế độ báo cáo của Giám đốc Ban Quản lý dự án ...,[243090],1
78,3414,Việc tích nước vận hành và tích nước thời kỳ t...,[65231],1
104,108965,Phanh cần trục khi dừng khẩn cấp phải là loại ...,[183501],1
106,103161,Ai có trách nhiệm báo cáo kết quả đánh giá ngh...,[177010],1
109,70371,Làm sao để được công nhận văn bằng nước ngoài ...,"[140295, 140296]",2


# Cell 12 - Tạo báo cáo split cuối cùng

## Mục tiêu cell này
Tổng hợp thông tin cuối cùng về các split sau khi xử lý leakage.

## Giải thích code
Code sẽ kiểm tra:
- số dòng của train strict, validation strict và test
- overlap `qid` giữa các split
- overlap exact question giữa các split
- coverage của relevant cid trong legal corpus

Sau đó lưu báo cáo vào:
`artifacts/leakage_reports/final_split_summary.csv`

## Output mong đợi
Bạn cần thấy:
- qid overlap = 0
- exact question overlap = 0
- cid coverage = 1.0 cho tất cả split

In [12]:
def exact_question_overlap(a, b):
    return len(set(a["question_norm"]) & set(b["question_norm"]))

def qid_overlap(a, b):
    return len(set(a["qid"]) & set(b["qid"]))

def coverage_rate(df):
    cids = get_cid_set(df)
    missing = cids - legal_corpus_cids
    return round((len(cids) - len(missing)) / len(cids), 6)

final_summary_rows = [
    {
        "split": "train_strict",
        "rows": len(train_strict_df),
        "unique_qid": train_strict_df["qid"].nunique(),
        "unique_questions": train_strict_df["question_norm"].nunique(),
        "unique_relevant_cids": len(get_cid_set(train_strict_df)),
        "cid_coverage": coverage_rate(train_strict_df)
    },
    {
        "split": "val_strict",
        "rows": len(val_strict_df),
        "unique_qid": val_strict_df["qid"].nunique(),
        "unique_questions": val_strict_df["question_norm"].nunique(),
        "unique_relevant_cids": len(get_cid_set(val_strict_df)),
        "cid_coverage": coverage_rate(val_strict_df)
    },
    {
        "split": "test",
        "rows": len(test_queries_df),
        "unique_qid": test_queries_df["qid"].nunique(),
        "unique_questions": test_queries_df["question_norm"].nunique(),
        "unique_relevant_cids": len(get_cid_set(test_queries_df)),
        "cid_coverage": coverage_rate(test_queries_df)
    }
]

final_split_summary_df = pd.DataFrame(final_summary_rows)

overlap_check_df = pd.DataFrame([
    {
        "check": "qid_overlap_train_val",
        "value": qid_overlap(train_strict_df, val_strict_df)
    },
    {
        "check": "qid_overlap_train_test",
        "value": qid_overlap(train_strict_df, test_queries_df)
    },
    {
        "check": "qid_overlap_val_test",
        "value": qid_overlap(val_strict_df, test_queries_df)
    },
    {
        "check": "exact_question_overlap_train_val",
        "value": exact_question_overlap(train_strict_df, val_strict_df)
    },
    {
        "check": "exact_question_overlap_train_test",
        "value": exact_question_overlap(train_strict_df, test_queries_df)
    },
    {
        "check": "exact_question_overlap_val_test",
        "value": exact_question_overlap(val_strict_df, test_queries_df)
    }
])

summary_path = REPORT_DIR / "final_split_summary.csv"
overlap_path = REPORT_DIR / "final_overlap_check.csv"

final_split_summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
overlap_check_df.to_csv(overlap_path, index=False, encoding="utf-8-sig")

print("Đã lưu:", summary_path)
print("Đã lưu:", overlap_path)

display(final_split_summary_df)
display(overlap_check_df)

Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\leakage_reports\final_split_summary.csv
Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\leakage_reports\final_overlap_check.csv


,split,rows,unique_qid,unique_questions,unique_relevant_cids,cid_coverage
0,train_strict,78812,78812,77864,51431,1.0
1,val_strict,8605,8605,8508,8109,1.0
2,test,29746,29746,29573,23907,1.0


,check,value
0,qid_overlap_train_val,0
1,qid_overlap_train_test,0
2,qid_overlap_val_test,0
3,exact_question_overlap_train_val,0
4,exact_question_overlap_train_test,0
5,exact_question_overlap_val_test,0


# Cell 13 - Lưu test final và tạo split manifest

## Mục tiêu cell này
Chốt bộ dữ liệu cuối cùng sau khi kiểm soát leakage.

## Giải thích code
Code sẽ:
- lưu `test_final.csv` vào `data/splits`
- tạo file `split_manifest.csv` mô tả rõ vai trò từng split
- kiểm tra lại các file split cuối cùng đã tồn tại

Các notebook sau sẽ dùng:
- `train_strict.csv`
- `val_strict.csv`
- `test_final.csv`

## Output mong đợi
Bạn cần thấy cả 4 file đều có trạng thái `OK`.

In [13]:
save_cols = ["qid", "question", "relevant_cids", "num_relevant"]

test_final_save = test_queries_df[save_cols].copy()
test_final_save["relevant_cids"] = test_final_save["relevant_cids"].apply(lambda xs: json.dumps(xs, ensure_ascii=False))

test_final_path = SPLIT_DIR / "test_final.csv"
test_final_save.to_csv(test_final_path, index=False, encoding="utf-8-sig")

split_manifest_df = pd.DataFrame([
    {
        "split": "train_strict",
        "file": "data/splits/train_strict.csv",
        "rows": len(train_strict_df),
        "purpose": "Dùng cho huấn luyện/tạo index phụ/tuning nếu cần, đã loại exact và near-duplicate với test"
    },
    {
        "split": "val_strict",
        "file": "data/splits/val_strict.csv",
        "rows": len(val_strict_df),
        "purpose": "Dùng để chọn top_k, chunk_size, threshold, reranker, không dùng test"
    },
    {
        "split": "test_final",
        "file": "data/splits/test_final.csv",
        "rows": len(test_queries_df),
        "purpose": "Chỉ dùng đánh giá cuối cùng"
    }
])

manifest_path = SPLIT_DIR / "split_manifest.csv"
split_manifest_df.to_csv(manifest_path, index=False, encoding="utf-8-sig")

required_split_files = [
    SPLIT_DIR / "train_strict.csv",
    SPLIT_DIR / "val_strict.csv",
    SPLIT_DIR / "test_final.csv",
    SPLIT_DIR / "split_manifest.csv"
]

check_df = pd.DataFrame([
    {
        "file": str(p.relative_to(PROJECT)),
        "status": "OK" if p.exists() else "MISSING",
        "size_mb": round(p.stat().st_size / 1024 / 1024, 2) if p.exists() else 0
    }
    for p in required_split_files
])

display(split_manifest_df)
display(check_df)

,split,file,rows,purpose
0,train_strict,data/splits/train_strict.csv,78812,Dùng cho huấn luyện/tạo index phụ/tuning nếu c...
1,val_strict,data/splits/val_strict.csv,8605,"Dùng để chọn top_k, chunk_size, threshold, rer..."
2,test_final,data/splits/test_final.csv,29746,Chỉ dùng đánh giá cuối cùng


,file,status,size_mb
0,data\splits\train_strict.csv,OK,10.70
1,data\splits\val_strict.csv,OK,1.17
2,data\splits\test_final.csv,OK,4.03
3,data\splits\split_manifest.csv,OK,0.00




## 1. File 02 dùng để làm gì?

File 02 không build RAG, không tạo index, không train model.

Nó chỉ làm nhiệm vụ:

```text
Kiểm tra dữ liệu có bị rò rỉ giữa train / validation / test không
và tạo bộ split sạch để các notebook sau sử dụng.
```

Nói dễ hiểu:

```text
Nếu câu hỏi trong test đã xuất hiện ở train,
hoặc câu gần giống test đã nằm trong train,
thì kết quả đánh giá sau này có thể bị ảo.
```

Vì vậy file 02 giúp mình chứng minh rằng:

```text
Test set được giữ sạch,
không dùng để chọn top_k,
không dùng để tune threshold,
không dùng để chọn reranker.
```

---

## 2. File 02 đã làm những bước nào?

### Bước 1: Load lại dữ liệu từ notebook 01

Bạn đã load:

```text
legal_corpus.csv                  68,663 dòng
train_queries.csv                 89,261 dòng
test_queries.csv                  29,746 dòng
company_handbook_documents.csv    16 dòng
```

Trong đó quan trọng nhất là:

```text
train_queries.csv
test_queries.csv
```

vì hai file này chứa:

```text
question
relevant_cids
num_relevant
```

---

## 3. Kiểm tra exact question overlap giữa train và test

Kết quả ban đầu:

```text
Exact overlap questions: 723
Train rows affected: 1017
Test rows affected: 797
```

Nghĩa là có **723 câu hỏi giống hệt nhau** xuất hiện ở cả train và test.

Đây là leakage nguy hiểm.

Ví dụ:

```text
Train có câu: "Viên chức là ai?"
Test cũng có câu: "Viên chức là ai?"
```

Nếu giữ nguyên thì sau này pipeline có thể “quen” câu hỏi này trong train, làm test score đẹp giả.

Cách xử lý của mình:

```text
Giữ nguyên test
Xóa các dòng train bị trùng với test
```

Kết quả sau xử lý:

```text
Train ban đầu: 89,261
Số dòng train bị loại: 1,017
Train còn lại: 88,244
Exact overlap còn lại: 0
```

Đây là bước rất quan trọng.

---

## 4. Kiểm tra duplicate question bên trong từng split

Kết quả:

```text
Duplicate questions trong train clean: 945
Duplicate questions trong test: 150
```

Cái này nghĩa là trong cùng một split vẫn có câu hỏi lặp lại.

Ví dụ trong train có nhiều dòng cùng câu:

```text
"Hợp đồng lao động là gì?"
```

Cái này chưa phải lỗi nghiêm trọng nếu vẫn nằm trong cùng một split.

Nhưng nó gây vấn đề khi chia validation. Nếu chia ngẫu nhiên theo dòng, cùng một câu hỏi có thể rơi vào cả train và validation.

Vì vậy mình không chia validation theo dòng, mà chia theo nhóm `question_norm`.

---

## 5. Tạo train/validation sạch theo nhóm câu hỏi

Sau khi chia theo `question_norm`, kết quả:

```text
Train final rows: 79,424
Validation rows: 8,820
Train/Val question overlap: 0
Val/Test question overlap: 0
```

Ý nghĩa:

```text
Không có câu hỏi nào xuất hiện đồng thời ở train và validation.
Không có câu hỏi nào xuất hiện đồng thời ở validation và test.
```

Đây là cách chia đúng hơn so với `train_test_split` ngẫu nhiên theo từng dòng.

---

## 6. Kiểm tra overlap `cid` giữa các split

Kết quả:

```text
train_final_vs_val: 4,170 cid overlap
train_final_vs_test: 10,494 cid overlap
val_vs_test: 2,698 cid overlap
```

Chỗ này cần hiểu đúng.

`cid overlap` nghĩa là cùng một đoạn văn bản pháp luật có thể được dùng để trả lời nhiều câu hỏi khác nhau ở các split khác nhau.

Ví dụ:

```text
Train hỏi: "Hợp đồng lao động là gì?"
Test hỏi: "Nội dung hợp đồng lao động gồm những gì?"
```

Hai câu khác nhau nhưng có thể cùng liên quan đến một văn bản luật.

Với RAG, **corpus tài liệu thường được dùng chung cho train/val/test**, nên mình không xóa `cid overlap` ngay. Mình chỉ ghi nhận vào report để minh bạch.

Kết luận phần này:

```text
Question overlap = nguy hiểm, phải xử lý.
CID overlap = cần báo cáo, nhưng không nhất thiết xóa vì corpus RAG được dùng chung.
```

---

## 7. Kiểm tra relevant cid có tồn tại trong legal corpus không

Kết quả:

```text
train_final coverage: 1.0
validation coverage: 1.0
test coverage: 1.0
```

Ý nghĩa:

```text
Mọi cid đúng trong train/validation/test đều tồn tại trong legal_corpus.csv.
```

Điều này cực quan trọng.

Nếu có `cid` không nằm trong corpus thì retriever không bao giờ tìm được đáp án đúng, làm metric sai.

---

## 8. Kiểm tra qid overlap

Kết quả:

```text
qid_overlap_train_val: 0
qid_overlap_train_test: 0
qid_overlap_val_test: 0
```

Nghĩa là không có mẫu dữ liệu nào bị trùng `qid` giữa các split.

Đây là split sạch.

---

## 9. Kiểm tra near-duplicate question

Kết quả:

```text
Near train/test: 612
Near val/test: 64
Near train/val: 181
```

Near-duplicate là câu không giống y hệt, nhưng rất gần nhau.

Ví dụ:

```text
Điều kiện để đăng ký kết hôn như thế nào?
Điều kiện để được đăng ký kết hôn như thế nào?
```

Chúng không trùng exact, nhưng gần như cùng ý.

Để chặt chẽ hơn, mình tạo split strict:

```text
Train final trước lọc: 79,424
Train strict sau lọc: 78,812
Train bị loại: 612

Validation trước lọc: 8,820
Validation strict sau lọc: 8,605
Validation bị loại: 215
```

Như vậy `train_strict` và `val_strict` sạch hơn.

---

## 10. Split cuối cùng dùng cho toàn bộ project

Cuối file 02, mình chốt 3 file chính:

```text
data/splits/train_strict.csv
data/splits/val_strict.csv
data/splits/test_final.csv
```

Vai trò từng file:

| File               | Vai trò                                                 |
| ------------------ | ------------------------------------------------------- |
| `train_strict.csv` | dùng cho train/tuning phụ nếu cần                       |
| `val_strict.csv`   | dùng để chọn `top_k`, `chunk_size`, threshold, reranker |
| `test_final.csv`   | chỉ dùng đánh giá cuối cùng                             |

Kết quả cuối:

```text
train_strict: 78,812 dòng
val_strict: 8,605 dòng
test_final: 29,746 dòng
```

Và kiểm tra cuối:

```text
qid overlap = 0
exact question overlap = 0
cid coverage = 1.0
```

---

## 11. Tại sao file 02 quan trọng?

Vì sau này khi mình báo cáo kết quả RAG, mình có thể nói:

```text
Chúng tôi không dùng test set để tune hệ thống.
Các câu hỏi trùng chính xác giữa train và test đã được loại khỏi train.
Validation được chia theo nhóm câu hỏi để tránh leakage.
Near-duplicate question được phát hiện và tạo strict split.
Test set chỉ dùng cho đánh giá cuối.
```

Đây là điểm làm bài của bạn nghiêm túc hơn rất nhiều.

## Chốt ngắn gọn

File 02 có nhiệm vụ:

```text
Làm sạch split
Kiểm tra leakage
Tạo train/val/test cuối cùng
Tạo report để đưa vào báo cáo
```

Sau file 02, dữ liệu đã đủ sạch để qua file 03:

```text
03_preprocessing_and_chunking.ipynb
```

Ở file 03 mình mới bắt đầu tiền xử lý text và chunk tài liệu để build RAG.
